# Project: Price Anomaly Hunter 🚨
Hourly DK1 electricity prices contain real oddities: negative prices on windy Sundays and spikes during cold, still evenings. Find them **without labels** using a robust rolling z-score and an Isolation Forest, then check both detectors on planted anomalies, where you *do* know the truth.

Topic: [[Anomaly Detection]] · guide note: [[Project - Price Anomaly Hunter]] · data as in [[Danish Electricity Prices]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, urllib.parse, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
f = pathlib.Path("data/Elspotprices_DK1_2024.csv"); f.parent.mkdir(exist_ok=True)
if not f.exists():
    q = {"start": "2024-01-01", "end": "2025-01-01", "filter": json.dumps({"PriceArea": ["DK1"]}), "limit": 0}
    pd.DataFrame(json.load(urllib.request.urlopen("https://api.energidataservice.dk/dataset/Elspotprices?" + urllib.parse.urlencode(q)))["records"]).to_csv(f, index=False)
p = pd.read_csv(f); y = p.assign(t=pd.to_datetime(p.HourUTC)).set_index("t").SpotPriceEUR.sort_index().asfreq("h")
print(f"{len(y)} hours of 2024, min {y.min():.0f}, max {y.max():.0f} EUR/MWh")

## 1. Robust rolling z-score
A plain z-score uses the mean and std, which the anomalies themselves distort. Use the **rolling median** and the **median absolute deviation** instead (centred window of `window` hours): $z = \frac{x - \text{med}}{1.4826\cdot\text{MAD}}$ (1.4826 makes the MAD match σ for Gaussian data).

In [ ]:
def robust_z(series, window=168):
    # TODO 1: rolling median, rolling MAD of the deviations, robust z-score (pandas Series in, Series out)
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_z():
    s = pd.Series(10 + np.random.default_rng(1).normal(0, 1, 101)); s.iloc[50] = 100.0
    z = robust_z(s, window=21)
    return z.iloc[50] > 50 and z.abs().drop(50).max() < 5
check("robust z", _t_z, "The planted 100 must get a huge z; ordinary noisy values must stay below 5.")

## 2. Precision and recall of a detector
Given the flagged timestamps and the true anomalies, return `(precision, recall)`.

In [ ]:
def precision_recall(flagged, truth):
    # TODO 2: precision = |flagged ∩ truth| / |flagged|, recall = |flagged ∩ truth| / |truth|
    raise NotImplementedError  # TODO 2

In [ ]:
check("precision/recall", lambda: precision_recall([1, 2, 3, 4], [3, 4, 5]) == (0.5, 2 / 3) and precision_recall([], [1]) == (0.0, 0.0), "{1,2,3,4} vs {3,4,5}: precision 2/4, recall 2/3.")
if ready("robust z", "precision/recall"):
    rng = np.random.default_rng(0); yp = y.copy()
    planted = rng.choice(np.arange(200, len(y) - 200), 30, replace=False)
    yp.iloc[planted] += rng.choice([-1, 1], 30) * rng.uniform(150, 300, 30)          # sensor-glitch style errors
    truth = set(yp.index[planted])
    K = 30                                                                    # alert budget: flag the 30 most suspicious hours
    flag_level = set(robust_z(yp).abs().nlargest(K).index)                    # z-score on the price itself
    spikiness = (yp - (yp.shift(1) + yp.shift(-1)) / 2).fillna(0)            # how much an hour sticks out from its neighbours
    flag_z = set(robust_z(spikiness).abs().nlargest(K).index)
    feats = pd.DataFrame({"price": yp, "diff_prev": yp.diff().fillna(0), "diff_next": (-yp.diff(-1)).fillna(0), "hour": yp.index.hour})
    iso = IsolationForest(contamination=30 / len(yp), random_state=0).fit(feats); flag_iso = set(yp.index[iso.predict(feats) == -1])
    for name, fl in [("z on price level", flag_level), ("z on spikiness", flag_z), ("Isolation Forest", flag_iso)]:
        pr, rc = precision_recall(fl, truth); print(f"{name:<17} flagged {len(fl):>3}  precision {pr:.2f}  recall {rc:.2f}")
    pr_z, rc_z = precision_recall(flag_z, truth)
    check("feature engineering wins", lambda: rc_z >= 0.6 and rc_z > precision_recall(flag_level, truth)[1],
          "The z-score on spikiness should catch ≥ 60 % of the glitches and beat the z-score on raw prices.")
    print("→ Same detector, better feature: glitches are sudden jumps, so measure jumpiness, not price level.")
    zr = robust_z(y); real = y[zr.abs() > 6]
    print(f"\nIn the REAL data, {len(real)} hours have |z| > 6, e.g.:"); print(real.sort_values().iloc[[0, 1, -2, -1]].round(1))
    plt.figure(figsize=(12, 3)); plt.plot(y.index, y, lw=.5); plt.scatter(real.index, real, c="red", s=8); plt.title("real 2024 prices, |robust z| > 6 in red"); plt.show()

Are the real-data hits errors? No: they're genuine market events (negative prices, scarcity spikes). **An anomaly detector finds the unusual, not the wrong.** Deciding which is which needs domain knowledge.

<details><summary>▶ Solution</summary>

```python
def robust_z(series, window=168):
    med = series.rolling(window, center=True, min_periods=1).median()
    mad = (series - med).abs().rolling(window, center=True, min_periods=1).median()
    return (series - med) / (1.4826 * mad + 1e-9)
```

```python
def precision_recall(flagged, truth):
    flagged = set(flagged); truth = set(truth)
    tp = len(flagged & truth)
    return tp / max(len(flagged), 1), tp / max(len(truth), 1)
```
</details>

## Stretch goals
- Add the wind forecast as a feature: are negative prices still anomalies once you know it was windy? (Conditional anomaly detection.)
- Try a seasonal decomposition (`statsmodels` STL) and flag large residuals.
- Plant *level shifts* (a week +50) instead of spikes. Which detector notices?